In [1]:
import os
import json
import warnings
import numpy as np
import torch
from tqdm import tqdm
from collections import defaultdict
from transformers import RobertaTokenizer, RobertaModel

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

DATA_PATH  = "../../dataset/devign/function.json"
OUTPUT_DIR = "./codebert"
MODEL_NAME = "microsoft/codebert-base"
MAX_LEN    = 512
BATCH_SIZE = 8

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

def load_dataset(path):
    print(f"Loading dataset from {path} ...")
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    grouped = defaultdict(list)
    for item in raw_data:
        grouped[item["project"]].append(item)
    print(f"Projects found: {list(grouped.keys())}")
    return grouped

def get_codebert_model():
    print(f"Loading CodeBERT model ({MODEL_NAME}) on {DEVICE} ...")
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
    model     = RobertaModel.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()
    return tokenizer, model

def mean_pool(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

def embed_code_snippets(snippets, tokenizer, model):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(snippets), BATCH_SIZE)):
            batch  = snippets[i:i + BATCH_SIZE]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LEN
            ).to(DEVICE)
            outputs = model(**inputs)
            pooled  = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            all_embeddings.append(pooled.cpu().numpy())
    return np.vstack(all_embeddings)

def save_embeddings(project, embeddings, labels):
    tag       = project.lower()
    emb_path  = os.path.join(OUTPUT_DIR, f"{tag}_embeddings.npy")
    lab_path  = os.path.join(OUTPUT_DIR, f"{tag}_labels.npy")
    meta_path = os.path.join(OUTPUT_DIR, f"{tag}_meta.json")
    np.save(emb_path, embeddings)
    np.save(lab_path, labels)
    meta = {"model": MODEL_NAME, "max_len": MAX_LEN, "pooling": "mean", "device": DEVICE}
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved {project}: {embeddings.shape} -> {emb_path}")

if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    combined_data    = load_dataset(DATA_PATH)
    tokenizer, model = get_codebert_model()

    for project_name, samples in combined_data.items():
        print(f"\nProcessing project: {project_name}")
        codes  = [s["func"] for s in samples]
        labels = np.array([s["target"] for s in samples])

        lengths   = [len(tokenizer.encode(c)) for c in codes]
        truncated = sum(1 for l in lengths if l > MAX_LEN)
        if truncated:
            print(f"{truncated}/{len(codes)} samples exceed {MAX_LEN} tokens and will be truncated")

        embeddings = embed_code_snippets(codes, tokenizer, model)
        save_embeddings(project_name, embeddings, labels)

    print("\nDone - all embeddings generated successfully.")

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset from ../../dataset/devign/function.json ...
Projects found: ['FFmpeg', 'qemu']
Loading CodeBERT model (microsoft/codebert-base) on mps ...


Token indices sequence length is longer than the specified maximum sequence length for this model (1123 > 512). Running this sequence through the model will result in indexing errors



Processing project: FFmpeg
6422/9769 samples exceed 512 tokens and will be truncated


100%|███████████████████████████████████████████████| 1222/1222 [11:17<00:00,  1.80it/s]


Saved FFmpeg: (9769, 768) -> ./ffmpeg_embeddings.npy

Processing project: qemu
6754/17549 samples exceed 512 tokens and will be truncated


100%|███████████████████████████████████████████████| 2194/2194 [22:18<00:00,  1.64it/s]

Saved qemu: (17549, 768) -> ./qemu_embeddings.npy

Done - all embeddings generated successfully.
